# Train, validate, test
**Machine Learning · Week 1 (Learning from examples) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w1d4.html

Measure a model honestly: learn on one part of the data, choose knobs on another,
and keep a third part locked away for the final exam.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=300, noise=0.35, random_state=1)

# first lock away the test set (20%), then cut the rest into training (60%) and validation (20%)
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)
print(f"train {len(X_train)} · validation {len(X_val)} · test {len(X_test)}")

## Choose k with the validation set

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

ks = list(range(1, 52, 2))
train_acc, val_acc = [], []
for k in ks:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    train_acc.append(model.score(X_train, y_train))
    val_acc.append(model.score(X_val, y_val))
best = ks[int(np.argmax(val_acc))]
print(f"best k on validation: {best}  (validation accuracy {max(val_acc):.1%})")

plt.figure(figsize=(6.4, 3.4))
plt.plot(ks, train_acc, "o-", color="#17a98c", label="training")
plt.plot(ks, val_acc, "o-", color="#7b5cfa", label="validation")
plt.axvline(best, color="#f0ac2e", ls="--", label=f"best k = {best}")
plt.xlabel("k")
plt.ylabel("accuracy")
plt.legend()
plt.title("Training accuracy flatters; validation tells the truth")
plt.show()

## The final exam, once
With `k` chosen, retrain on training + validation (more data is better) and score the test set a single time.

In [ ]:
final = KNeighborsClassifier(n_neighbors=best).fit(X_rest, y_rest)
print(f"test accuracy with k = {best}: {final.score(X_test, y_test):.1%}")

## 5-fold cross-validation
Every point outside the test set takes one turn in the validation fold.

In [ ]:
from sklearn.model_selection import cross_val_score

for k in [1, best, 51]:
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_rest, y_rest, cv=5)
    print(f"k = {k:2d}: folds {np.round(scores, 2)}  ->  {scores.mean():.1%} ± {scores.std():.1%}")

## Your turn: noisier data, bigger k?
Fill in `best_k`. It should return the `k` with the best validation accuracy.

In [ ]:
def best_k(noise, seed=0):
    X, y = make_moons(n_samples=300, noise=noise, random_state=seed)
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    # TODO: try k = 1, 3, 5, ..., 51 and return the one with the best validation accuracy
    return 1

for noise in [0.1, 0.3, 0.5]:
    print(f"noise {noise}: best k = {best_k(noise)}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def best_k(noise, seed=0):
    X, y = make_moons(n_samples=300, noise=noise, random_state=seed)
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    ks = range(1, 52, 2)
    scores = [KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr).score(X_va, y_va) for k in ks]
    return list(ks)[int(np.argmax(scores))]

# one split can be lucky, so look at 10 different splits for each noise level
for noise in [0.1, 0.3, 0.5]:
    picks = [best_k(noise, seed) for seed in range(10)]
    print(f"noise {noise}: best k over 10 splits = {picks}  (median {int(np.median(picks))})")